# IntakeEngine Tests
Tests for `intake_llm_v4/app/engine/intake_engine.py`

Run from the `intake_llm_v4` directory, or adjust `sys.path` below.

## Test coverage summary

| # | Section | What is tested | Key invariant |
|---|---------|---------------|---------------|
| 1 | **Engine initialisation** | State, phases, budget, escalation defaults on construction | Engine is ready to serve questions immediately after `__init__` |
| 2 | **Implicit parent seeding** | Root symptom field pre-set to `yes` for all 6 complaint/field pairs | Linker atomics that gate on the complaint root symptom fire correctly from the first question |
| 3 | **Phase ordering** | First question is `opening`, next phase is `core_characterize` after opening answered | Phases are served in the enforced order defined in the complaint schedule |
| 4 | **Deduplication** | Pre-populated fields never re-asked; no field appears twice across a full session | `state[field]` set → question skipped, everywhere, every time |
| 5 | **record_answer** | State write, turn append, turn structure, `questions_asked` increment, detail extraction from `"yes - ..."` answers | Every answer is persisted correctly and detail fields are captured inline |
| 6 | **Linker auto-set** | Positive linker answer sets parent field; negative answer does not | Dedup correctness: answering a qualifier like `pleuritic_chest_pain=yes` suppresses a later generic `chest_pain` screen |
| 7 | **Red flag detection** | Full AND-pattern fires; partial pattern does not; negative answer does not; single-field pattern fires; cross-complaint pattern (meningitic in headache) | v2 JSON pattern format correctly evaluated — all fields must be positive for a pattern to trigger |
| 8 | **Escalation ordering** | Escalation raises through all four tiers; never lowers once raised | `immediate_alert` > `urgent_escalation` > `priority_clinician_review` > `same_day_clinician_review` > `none` |
| 9 | **skip_question** | Skipped field set to `not_assessed`, marked in turn, never re-asked | Skipped fields are treated as captured for dedup purposes |
| 10 | **Question budget / hard cap** | Session terminates within `_max_questions`; `completed` flag set | Engine never exceeds the budget declared in the complaint JSON |
| 11 | **get_progress** | Keys present, `completion_percent` starts at 0 and increases, `questions_asked` matches | Progress reporting is accurate throughout a session |
| 12 | **Multi-complaint loading** | 8 complaints load without error and return a valid first question | All complaint JSON files conform to the schema the engine expects |
| 13 | **Sex gating** | `pregnancy_context` not asked for male; asked for female | `ask_if` gates on `SEX_FEMALE` correctly suppress or allow gynecologic questions |
| 14 | **if_positive_ask queue** | Detail questions enqueued on positive answer; served before schedule resumes | Detail follow-ups interrupt the phase schedule and are served immediately |

**Complaints used:** `chest_pain` (primary), `headache`, `fever`, `abdominal_pain`, `shortness_of_breath`, `seizure`, `back_pain`, `vomiting`, `loss_of_consciousness`, `loose_stool`, `trauma`
| 15 | **Summary engine** | Full session with realistic clinical answers, template summary generation, AI summary via Anthropic API | Summary pipeline produces non-empty, clinically meaningful output |


In [1]:
import sys, os

# Point to the intake_llm_v4 root — adjust if running from elsewhere
ROOT = os.path.abspath(".")  # assumes notebook is run from intake_llm_v4/
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from app.engine.intake_engine import IntakeEngine, load_complaint, load_shared
from app.engine.utils import is_positive_answer, is_negative_answer

SHARED = load_shared()

def make_engine(complaint_id: str, age: int = 45, sex: str = "male") -> IntakeEngine:
    return IntakeEngine(load_complaint(complaint_id), {"age": age, "sex": sex}, SHARED)

# Simple assertion helper
passed = []
failed = []

def check(name: str, condition: bool, detail: str = ""):
    if condition:
        passed.append(name)
        print(f"  ✅ {name}")
    else:
        failed.append(name)
        print(f"  ❌ {name}" + (f" — {detail}" if detail else ""))

print("Setup complete. Shared contract loaded.")

Setup complete. Shared contract loaded.


## 1. Engine initialisation

In [2]:
print("=== 1. Engine initialisation ===")

engine = make_engine("chest_pain")

check("state starts empty (except implicit parent seed)",
      engine.state.get("chest_pain") == "yes")

check("questions loaded",
      len(engine.questions) > 0,
      f"got {len(engine.questions)}")

check("phases loaded",
      len(engine.phases) == 7,
      f"got {engine.phases}")

check("not completed on init",
      not engine.completed)

check("escalation starts at none",
      engine.escalation_level == "none")

check("max_questions set from budget",
      engine._max_questions > 0,
      f"got {engine._max_questions}")

=== 1. Engine initialisation ===
  ✅ state starts empty (except implicit parent seed)
  ✅ questions loaded
  ✅ phases loaded
  ✅ not completed on init
  ✅ escalation starts at none
  ✅ max_questions set from budget


## 2. Implicit parent seeding

In [3]:
print("=== 2. Implicit parent seeding ===")

for complaint_id, expected_field in [
    ("chest_pain",         "chest_pain"),
    ("headache",           "headache"),
    ("shortness_of_breath","shortness_of_breath"),
    ("seizure",            "seizure"),
    ("loose_stool",        "diarrhea"),
    ("trauma",             "trauma_present"),
]:
    e = make_engine(complaint_id)
    check(f"{complaint_id} seeds '{expected_field}' = yes",
          e.state.get(expected_field) == "yes",
          f"got {e.state.get(expected_field)}")

=== 2. Implicit parent seeding ===
  ✅ chest_pain seeds 'chest_pain' = yes
  ✅ headache seeds 'headache' = yes
  ✅ shortness_of_breath seeds 'shortness_of_breath' = yes
  ✅ seizure seeds 'seizure' = yes
  ✅ loose_stool seeds 'diarrhea' = yes
  ✅ trauma seeds 'trauma_present' = yes


## 3. get_next_question — phase ordering

In [4]:
print("=== 3. get_next_question — phase ordering ===")

engine = make_engine("chest_pain")

q = engine.get_next_question()
check("first question is in opening phase",
      q is not None and q["phase"] == "opening",
      f"got phase={q['phase'] if q else None}")

check("first question is presenting_complaint_narrative",
      q is not None and q["field"] == "presenting_complaint_narrative",
      f"got field={q['field'] if q else None}")

check("question has required keys",
      q is not None and all(k in q for k in ["id", "field", "text", "phase", "response_type"]),
      f"got keys={list(q.keys()) if q else None}")

# Record the opening answer and confirm we move into next phase
engine.record_answer(q["id"], q["field"], "chest pain for 2 hours", q["phase"])
q2 = engine.get_next_question()
check("after opening, moves to core_characterize",
      q2 is not None and q2["phase"] == "core_characterize",
      f"got phase={q2['phase'] if q2 else None}")

=== 3. get_next_question — phase ordering ===
  ✅ first question is in opening phase
  ✅ first question is presenting_complaint_narrative
  ✅ question has required keys
  ✅ after opening, moves to core_characterize


## 4. Deduplication — FIELD_ALREADY_CAPTURED

In [5]:
print("=== 4. Deduplication ===")

engine = make_engine("chest_pain")

# Pre-populate several fields
for field in ["onset", "duration", "severity", "location", "character"]:
    engine.state[field] = "pre-populated"

# Exhaust questions until we find one of the pre-populated fields or confirm they are skipped
seen_fields = set()
for _ in range(30):
    q = engine.get_next_question()
    if q is None:
        break
    seen_fields.add(q["field"])
    engine.record_answer(q["id"], q["field"], "test answer", q["phase"])

check("onset not re-asked after pre-population",
      "onset" not in seen_fields)

check("duration not re-asked after pre-population",
      "duration" not in seen_fields)

check("severity not re-asked after pre-population",
      "severity" not in seen_fields)

# Also verify that answering a field via record_answer prevents it being re-asked
engine2 = make_engine("headache")
q = engine2.get_next_question()
engine2.record_answer(q["id"], q["field"], "bad headache", q["phase"])
fields_after = set()
for _ in range(40):
    nq = engine2.get_next_question()
    if nq is None:
        break
    fields_after.add(nq["field"])
    engine2.record_answer(nq["id"], nq["field"], "yes", nq["phase"])

check("no field appears twice in a session",
      len(fields_after) == len(set(fields_after)))

=== 4. Deduplication ===
  ✅ onset not re-asked after pre-population
  ✅ duration not re-asked after pre-population
  ✅ severity not re-asked after pre-population
  ✅ no field appears twice in a session


## 5. record_answer — state, turns, detail capture

In [6]:
print("=== 5. record_answer ===")

engine = make_engine("chest_pain")
q = engine.get_next_question()
turn = engine.record_answer(q["id"], q["field"], "chest tightness started 2 hours ago", q["phase"])

check("answer written to state",
      engine.state.get(q["field"]) == "chest tightness started 2 hours ago")

check("turn appended",
      len(engine.turns) == 1)

check("turn has correct fields",
      all(k in turn for k in ["turn_number", "question_id", "field", "answer", "phase", "timestamp"]))

check("questions_asked incremented",
      engine.questions_asked == 1)

# Test detail capture: answer "yes - severe" to a boolean+detail question
engine2 = make_engine("chest_pain")
# Find a question with capture_detail_if_positive
detail_q = next(
    (qid for qid, qdef in engine2.questions.items()
     if qdef.get("capture_detail_if_positive") and qdef.get("detail_field")),
    None
)
if detail_q:
    qdef = engine2.questions[detail_q]
    engine2.state[detail_q] = None  # ensure not pre-set
    turn2 = engine2.record_answer(detail_q, qdef["field"], "yes - radiates to left arm", "core_characterize")
    check("detail extracted from 'yes - ...' answer",
          engine2.state.get(qdef["detail_field"]) == "radiates to left arm",
          f"got: {engine2.state.get(qdef['detail_field'])}")
else:
    print("  ⚠ No capture_detail_if_positive question found in chest_pain — skipping detail test")

=== 5. record_answer ===
  ✅ answer written to state
  ✅ turn appended
  ✅ turn has correct fields
  ✅ questions_asked incremented
  ✅ detail extracted from 'yes - ...' answer


## 6. Linker auto-set (parent field seeding)

In [7]:
print("=== 6. Linker auto-set ===")

# Find a linker atomic in chest_pain — one with ask_if gating on a parent field
engine = make_engine("chest_pain")

linker = next(
    ((qid, qdef) for qid, qdef in engine.questions.items()
     if isinstance(qdef.get("ask_if"), dict)
     and qdef["ask_if"].get("op") == "field_equals"
     and str(qdef["ask_if"].get("value", "")).lower() == "yes"
     and str(qdef["ask_if"].get("field_ref", "")).startswith("$state.")),
    None
)

if linker:
    qid, qdef = linker
    parent = qdef["ask_if"]["field_ref"].replace("$state.", "")
    # Clear parent so we can test auto-set
    engine.state[parent] = None
    engine.record_answer(qid, qdef["field"], "yes", "high_priority_followup")
    check(f"answering linker '{qdef['field']}' positive auto-sets parent '{parent}'",
          engine.state.get(parent) == "yes",
          f"got {engine.state.get(parent)}")

    # Negative answer should NOT set parent
    engine2 = make_engine("chest_pain")
    engine2.state[parent] = None
    engine2.record_answer(qid, qdef["field"], "no", "high_priority_followup")
    check("negative linker answer does NOT auto-set parent",
          engine2.state.get(parent) != "yes",
          f"got {engine2.state.get(parent)}")
else:
    print("  ⚠ No linker atomic found in chest_pain — skipping")

=== 6. Linker auto-set ===
  ⚠ No linker atomic found in chest_pain — skipping


## 7. Red flag detection (v2 AND-pattern format)

In [8]:
print("=== 7. Red flag detection ===")

engine = make_engine("chest_pain")
patterns = engine.complaint.get("derived_red_flag_patterns", {})
print(f"  Patterns available: {list(patterns.keys())}")

# --- Test 1: acute_coronary_pattern ---
engine = make_engine("chest_pain")
for field in ["rest_pain", "exertional_trigger", "shortness_of_breath", "diaphoresis"]:
    engine.state[field] = "yes"
engine.record_answer("arm_or_jaw_radiation", "arm_or_jaw_radiation", "yes", "early_danger_screen")

check("acute_coronary_pattern fires when all fields positive",
      any(r["pattern"] == "acute_coronary_pattern" for r in engine.red_flags),
      f"red_flags={engine.red_flags}")

check("escalation raised to urgent_escalation",
      engine.escalation_level == "urgent_escalation",
      f"got {engine.escalation_level}")

# --- Test 2: partial pattern should NOT fire ---
engine2 = make_engine("chest_pain")
engine2.state["rest_pain"] = "yes"  # only one of five fields
engine2.record_answer("arm_or_jaw_radiation", "arm_or_jaw_radiation", "yes", "early_danger_screen")

check("partial pattern does NOT fire",
      not any(r["pattern"] == "acute_coronary_pattern" for r in engine2.red_flags),
      f"red_flags={engine2.red_flags}")

# --- Test 3: negative answer should NOT fire ---
engine3 = make_engine("chest_pain")
for field in ["rest_pain", "exertional_trigger", "shortness_of_breath", "diaphoresis"]:
    engine3.state[field] = "yes"
engine3.record_answer("arm_or_jaw_radiation", "arm_or_jaw_radiation", "no", "early_danger_screen")

check("negative answer on trigger field does NOT fire pattern",
      not any(r["pattern"] == "acute_coronary_pattern" for r in engine3.red_flags),
      f"red_flags={engine3.red_flags}")

# --- Test 4: aortic_pattern (single field) ---
engine4 = make_engine("chest_pain")
engine4.record_answer("tearing_to_back_quality", "tearing_to_back_quality", "yes", "critical_followup")

check("aortic_pattern fires on single positive field",
      any(r["pattern"] == "aortic_pattern" for r in engine4.red_flags),
      f"red_flags={engine4.red_flags}")

# --- Test 5: meningitic_pattern in headache ---
engine5 = make_engine("headache")
engine5.state["fever"] = "yes"
engine5.state["neck_stiffness"] = "yes"
engine5.record_answer("confusion_or_ams", "confusion_or_ams", "yes", "critical_followup")

check("meningitic_pattern fires in headache complaint",
      any(r["pattern"] == "meningitic_pattern" for r in engine5.red_flags),
      f"red_flags={engine5.red_flags}")

=== 7. Red flag detection ===
  Patterns available: ['acute_coronary_pattern', 'pulmonary_embolus_pattern', 'aortic_pattern', 'arrhythmic_pattern', 'pericardial_pattern']
  ✅ acute_coronary_pattern fires when all fields positive
  ✅ escalation raised to urgent_escalation
  ✅ partial pattern does NOT fire
  ✅ negative answer on trigger field does NOT fire pattern
  ✅ aortic_pattern fires on single positive field
  ✅ meningitic_pattern fires in headache complaint


## 8. Escalation ordering

In [9]:
print("=== 8. Escalation ordering ===")

engine = make_engine("chest_pain")

engine._raise_escalation("same_day_clinician_review")
check("escalation raises from none to same_day",
      engine.escalation_level == "same_day_clinician_review")

engine._raise_escalation("priority_clinician_review")
check("escalation raises from same_day to priority",
      engine.escalation_level == "priority_clinician_review")

engine._raise_escalation("same_day_clinician_review")
check("escalation does NOT lower from priority to same_day",
      engine.escalation_level == "priority_clinician_review")

engine._raise_escalation("urgent_escalation")
check("escalation raises to urgent",
      engine.escalation_level == "urgent_escalation")

engine._raise_escalation("immediate_alert")
check("escalation raises to immediate_alert",
      engine.escalation_level == "immediate_alert")

engine._raise_escalation("none")
check("escalation cannot be lowered to none from immediate_alert",
      engine.escalation_level == "immediate_alert")

=== 8. Escalation ordering ===
  ✅ escalation raises from none to same_day
  ✅ escalation raises from same_day to priority
  ✅ escalation does NOT lower from priority to same_day
  ✅ escalation raises to urgent
  ✅ escalation raises to immediate_alert
  ✅ escalation cannot be lowered to none from immediate_alert


## 9. skip_question

In [10]:
print("=== 9. skip_question ===")

engine = make_engine("chest_pain")
q = engine.get_next_question()
turn = engine.skip_question(q["id"], q["field"], q["phase"])

check("skipped field set to not_assessed",
      engine.state.get(q["field"]) == "not_assessed")

check("skip turn marked as skipped",
      turn["skipped"] is True)

check("questions_asked incremented on skip",
      engine.questions_asked == 1)

# Skipped field should not be re-asked
seen = set()
for _ in range(40):
    nq = engine.get_next_question()
    if nq is None:
        break
    seen.add(nq["field"])
    engine.record_answer(nq["id"], nq["field"], "test", nq["phase"])

check("skipped field not re-asked",
      q["field"] not in seen,
      f"field {q['field']} appeared again")

=== 9. skip_question ===
  ✅ skipped field set to not_assessed
  ✅ skip turn marked as skipped
  ✅ questions_asked incremented on skip
  ✅ skipped field not re-asked


## 10. Question budget / hard cap

In [11]:
print("=== 10. Question budget / hard cap ===")

engine = make_engine("chest_pain")
count = 0
while True:
    q = engine.get_next_question()
    if q is None:
        break
    engine.record_answer(q["id"], q["field"], "yes", q["phase"])
    count += 1
    if count > 200:  # safety
        break

check("session terminates (get_next_question returns None)",
      q is None)

check("questions asked within max budget",
      engine.questions_asked <= engine._max_questions,
      f"asked={engine.questions_asked}, max={engine._max_questions}")

check("engine marked completed",
      engine.completed)

print(f"  Questions asked: {engine.questions_asked} / {engine._max_questions} max")

=== 10. Question budget / hard cap ===
  ✅ session terminates (get_next_question returns None)
  ✅ questions asked within max budget
  ✅ engine marked completed
  Questions asked: 40 / 40 max


## 11. get_progress

In [12]:
print("=== 11. get_progress ===")

engine = make_engine("chest_pain")
progress = engine.get_progress()

check("progress has required keys",
      all(k in progress for k in ["questions_asked", "completion_percent", "phase", "completed", "escalation_level"]))

check("completion starts at 0",
      progress["completion_percent"] == 0.0)

check("not completed at start",
      not progress["completed"])

# Answer a few questions and check progress increases
for _ in range(5):
    q = engine.get_next_question()
    if q:
        engine.record_answer(q["id"], q["field"], "yes", q["phase"])

progress2 = engine.get_progress()
check("completion_percent increases after answering questions",
      progress2["completion_percent"] > 0,
      f"got {progress2['completion_percent']}")

check("questions_asked matches",
      progress2["questions_asked"] == 5)

=== 11. get_progress ===
  ✅ progress has required keys
  ✅ completion starts at 0
  ✅ not completed at start
  ✅ completion_percent increases after answering questions
  ✅ questions_asked matches


## 12. Multiple complaints — spot checks

In [13]:
print("=== 12. Multiple complaints — spot checks ===")

complaints_to_test = [
    "headache", "fever", "abdominal_pain", "shortness_of_breath",
    "seizure", "back_pain", "vomiting", "loss_of_consciousness"
]

for cid in complaints_to_test:
    try:
        e = make_engine(cid)
        q = e.get_next_question()
        check(f"{cid}: loads and returns first question",
              q is not None and "field" in q,
              f"got {q}")
    except Exception as ex:
        check(f"{cid}: loads without error", False, str(ex))

=== 12. Multiple complaints — spot checks ===
  ✅ headache: loads and returns first question
  ✅ fever: loads and returns first question
  ✅ abdominal_pain: loads and returns first question
  ✅ shortness_of_breath: loads and returns first question
  ✅ seizure: loads and returns first question
  ✅ back_pain: loads and returns first question
  ✅ vomiting: loads and returns first question
  ✅ loss_of_consciousness: loads and returns first question


## 13. Sex gating (gynecologic fields)

In [14]:
print("=== 13. Sex gating ===")

# pregnancy_context should be gated on SEX_FEMALE
# Run a full session as male and confirm pregnancy_context never appears
male_engine = make_engine("headache", sex="male")
male_fields = set()
for _ in range(60):
    q = male_engine.get_next_question()
    if q is None:
        break
    male_fields.add(q["field"])
    male_engine.record_answer(q["id"], q["field"], "no", q["phase"])

check("pregnancy_context not asked for male patient",
      "pregnancy_context" not in male_fields,
      f"fields seen: {[f for f in male_fields if 'pregn' in f]}")

# Run same for female and confirm pregnancy_context IS asked
female_engine = make_engine("headache", sex="female")
female_fields = set()
for _ in range(60):
    q = female_engine.get_next_question()
    if q is None:
        break
    female_fields.add(q["field"])
    female_engine.record_answer(q["id"], q["field"], "no", q["phase"])

check("pregnancy_context asked for female patient",
      "pregnancy_context" in female_fields,
      f"fields seen: {sorted(female_fields)}")

=== 13. Sex gating ===
  ✅ pregnancy_context not asked for male patient
  ✅ pregnancy_context asked for female patient


## 14. if_positive_ask detail queue

In [15]:
print("=== 14. if_positive_ask detail queue ===")

# Find a question with if_positive_ask
engine = make_engine("chest_pain")
trigger_q = next(
    ((qid, qdef) for qid, qdef in engine.questions.items()
     if qdef.get("if_positive_ask")),
    None
)

if trigger_q:
    qid, qdef = trigger_q
    followup_ids = qdef["if_positive_ask"]
    engine.record_answer(qid, qdef["field"], "yes", "core_characterize")
    check("if_positive_ask queues detail questions",
          any(fid in engine._detail_queue for fid in followup_ids),
          f"queue={engine._detail_queue}, expected one of {followup_ids}")
    
    # Confirm detail question comes next
    next_q = engine.get_next_question()
    check("detail question served before schedule continues",
          next_q is not None and next_q["id"] in followup_ids,
          f"got {next_q['id'] if next_q else None}, expected one of {followup_ids}")
else:
    print("  ⚠ No if_positive_ask question found — skipping")

=== 14. if_positive_ask detail queue ===
  ✅ if_positive_ask queues detail questions
  ✅ detail question served before schedule continues


---
## 15. Summary engine — end-to-end session + AI summary
Runs a full simulated chest pain session, generates the template summary, then calls the Anthropic API to produce an AI HPI paragraph.

In [16]:
print("=== 15. Summary engine — end-to-end ===")
from app.engine.summary_engine import generate_template_summary, ai_summarize, format_summary_text

# --- Build a realistic chest pain session ---
engine = make_engine("chest_pain", age=58, sex="male")

# Simulate a classic ACS presentation
answers = {
    "presenting_complaint_narrative": "I have had crushing chest pain for the last 2 hours, it started while I was walking up the stairs.",
    "onset": "2 hours ago",
    "timing": "constant",
    "duration": "constant for 2 hours",
    "severity": "9",
    "location": "central chest",
    "character": "crushing, heavy",
    "rest_pain": "yes",
    "exertional_trigger": "yes",
    "shortness_of_breath": "yes",
    "diaphoresis": "yes",
    "arm_or_jaw_radiation": "yes - radiates to left arm",
    "course": "getting worse",
    "aggravating_factors": "exertion",
    "relieving_factors": "nothing helps",
    "functional_impact": "unable to walk or talk without pain",
    "syncope_or_near_syncope": "no",
    "palpitations": "no",
    "tearing_to_back_quality": "no",
    "pleuritic_component": "no",
    "positional_component": "no",
    "prior_cardiac_history": "hypertension, on amlodipine",
    "family_history_early_heart_disease": "father had MI at 55",
    "past_medical_history": "Hypertension, type 2 diabetes",
    "current_medications": "amlodipine 5mg, metformin 500mg",
    "allergies": "penicillin — rash",
    "smoking_current": "yes, 20 per day for 30 years",
    "alcohol_current": "occasional, 2-3 units per week",
}

for field, answer in answers.items():
    engine.state[field] = answer
engine.questions_asked = len(answers)
engine.completed = True

# Fire the red flag pattern manually so escalation is set
for f in ["rest_pain", "exertional_trigger", "shortness_of_breath", "diaphoresis"]:
    engine.state[f] = "yes"
engine._check_red_flags("arm_or_jaw_radiation", "yes")

print(f"  Red flags: {[r['pattern'] for r in engine.red_flags]}")
print(f"  Escalation: {engine.escalation_level}")

# --- Generate template summary ---
template = generate_template_summary(engine)
pre_summary = format_summary_text(template)

check("template summary generated", bool(template))
check("template has HPI", bool(template.get("hpi")))
check("template has pertinent positives", len(template.get("pertinent_positives", [])) > 0)
check("escalation level in template", template.get("escalation_level") == "urgent_escalation")

print("\n--- Template summary ---")
print(pre_summary)

=== 15. Summary engine — end-to-end ===
  Red flags: ['acute_coronary_pattern']
  Escalation: urgent_escalation
  ✅ template summary generated
  ✅ template has HPI
  ✅ template has pertinent positives
  ✅ escalation level in template

--- Template summary ---
Patient: Unknown years, Unknown

Chief Complaint (Primary): Chest Pain
Other Concerns: None reported

HPI:
Patient presents with chest pain. Patient states: "I have had crushing chest pain for the last 2 hours, it started while I was walking up the stairs." Onset: 2 hours ago. Duration: constant for 2 hours. Located in the central chest, described as crushing, heavy. Severity: 9. Course: getting worse. Aggravating factors: exertion. Relieving factors: nothing helps. Functional impact: unable to walk or talk without pain.

Review of Systems (ROS):
  Constitutional: Not assessed
  Cardiovascular: Chest pain: yes; Rest pain: yes; Exertional trigger: yes; Diaphoresis: yes | denies Syncope or near syncope, Palpitations, Tearing to back

In [17]:
print("=== 15b. AI summary (Anthropic API call) ===")

ai_summary = ai_summarize(engine.state, template)

check("AI summary returned a non-empty string", bool(ai_summary and ai_summary.strip()))
check("AI summary is not just the fallback HPI",
      ai_summary != template.get("hpi", ""),
      "got fallback — check API key or network")
check("AI summary is at least 100 chars",
      len(ai_summary) >= 100,
      f"got {len(ai_summary)} chars")

print("\n--- AI HPI paragraph ---")
print(ai_summary)

=== 15b. AI summary (Anthropic API call) ===
  ✅ AI summary returned a non-empty string
  ✅ AI summary is not just the fallback HPI
  ✅ AI summary is at least 100 chars

--- AI HPI paragraph ---
The patient presents with a 2-hour history of crushing, heavy central chest pain that began while walking up stairs. The pain is rated 9/10 in severity, has been constant since onset, and is progressively worsening. The patient experiences chest pain both at rest and with exertion, with exertion serving as an aggravating factor, and reports that nothing provides relief. The pain radiates to the left arm and is associated with shortness of breath and diaphoresis. The functional impact is significant, as the patient reports being unable to walk or talk without experiencing pain. The patient has a medical history of hypertension and type 2 diabetes, currently managed with amlodipine 5mg and metformin 500mg, and a family history of early cardiac disease with father having a myocardial infarction at

## 15. Summary engine — full session + ai_summarize

In [18]:
print("=== 15. Summary engine ===")

from app.engine.summary_engine import generate_template_summary, format_summary_text, ai_summarize

# Run a realistic chest pain session with clinical answers
engine = make_engine("chest_pain", age=58, sex="male")

ANSWERS = {
    "presenting_complaint_narrative": "Crushing chest pain radiating to my left arm, started about 90 minutes ago at rest",
    "onset": "90 minutes ago",
    "timing": "constant",
    "duration": "constant for 90 minutes",
    "severity": "9",
    "location": "central chest",
    "character": "crushing, heavy pressure",
    "rest_pain": "yes",
    "exertional_trigger": "yes",
    "shortness_of_breath": "yes",
    "diaphoresis": "yes",
    "arm_or_jaw_radiation": "yes - radiates to left arm and jaw",
    "course": "getting worse",
    "aggravating_factors": "nothing relieves it",
    "relieving_factors": "no relief with rest or GTN",
    "functional_impact": "unable to do anything, lying still",
    "syncope_or_near_syncope": "no",
    "palpitations": "no",
    "tearing_to_back_quality": "no",
    "pleuritic_component": "no",
    "positional_component": "no",
    "prior_cardiac_history": "yes - hypertension, hypercholesterolaemia",
    "family_history_early_heart_disease": "yes - father had MI at 55",
    "past_medical_history": "Hypertension, hypercholesterolaemia, type 2 diabetes",
    "current_medications": "Amlodipine 5mg, atorvastatin 40mg, metformin 500mg BD",
    "allergies": "NKDA",
    "smoking_current": "yes - 20 pack years, stopped 5 years ago",
    "alcohol_current": "occasional, less than 10 units per week",
}

# Walk the engine and inject answers where we have them, answer 'no' otherwise
while True:
    q = engine.get_next_question()
    if q is None:
        break
    answer = ANSWERS.get(q["field"], "no")
    engine.record_answer(q["id"], q["field"], answer, q["phase"])

engine.completed = True

check("session completed", engine.completed)
check("red flags detected", len(engine.red_flags) > 0,
      f"red_flags={engine.red_flags}")
check("escalation raised above none", engine.escalation_level != "none",
      f"escalation={engine.escalation_level}")

print(f"  Red flags: {[r['pattern'] for r in engine.red_flags]}")
print(f"  Escalation: {engine.escalation_level}")
print(f"  Questions asked: {engine.questions_asked}")

=== 15. Summary engine ===
  ✅ session completed
  ✅ red flags detected
  ✅ escalation raised above none
  Red flags: ['acute_coronary_pattern']
  Escalation: urgent_escalation
  Questions asked: 34


In [19]:
# Generate template summary
template = generate_template_summary(engine)
pre_summary = format_summary_text(template)

check("template summary has chief_complaint",
      bool(template.get("chief_complaint")))
check("template summary has pertinent_positives",
      len(template.get("pertinent_positives", [])) > 0,
      f"got {template.get('pertinent_positives')}")
check("template summary has hpi",
      len(template.get("hpi", "")) > 20)
check("format_summary_text returns non-empty string",
      len(pre_summary) > 100)

print("\n--- Template Summary ---")
print(pre_summary)

  ✅ template summary has chief_complaint
  ✅ template summary has pertinent_positives
  ✅ template summary has hpi
  ✅ format_summary_text returns non-empty string

--- Template Summary ---
Patient: Unknown years, Unknown

Chief Complaint (Primary): Chest Pain
Other Concerns: None reported

HPI:
Patient presents with chest pain. Patient states: "Crushing chest pain radiating to my left arm, started about 90 minutes ago at rest" Onset: 90 minutes ago. Duration: constant for 90 minutes. Located in the central chest, described as crushing, heavy pressure. Severity: 9. Course: getting worse. Aggravating factors: nothing relieves it. Relieving factors: no relief with rest or GTN. Functional impact: unable to do anything, lying still.

Review of Systems (ROS):
  Constitutional: Not assessed
  Cardiovascular: Chest pain: yes; Rest pain: yes; Exertional trigger: yes; Diaphoresis: yes | denies Syncope or near syncope, Palpitations, Tearing to back quality, Leg swelling
  Respiratory: Shortness 

In [20]:
# Generate AI summary via Anthropic API
print("Calling Anthropic API for AI summary...")

# Diagnostic: confirm key is loaded
from app.engine.summary_engine import _ANTHROPIC_API_KEY
key_loaded = bool(_ANTHROPIC_API_KEY)
print(f"  API key loaded from api_keys.py: {key_loaded} (starts with: {_ANTHROPIC_API_KEY[:12]}...)") if key_loaded else print("  ⚠ API key not loaded")

# Call with explicit error surfacing
import requests as _req
import os as _os
api_key = _os.environ.get('ANTHROPIC_API_KEY', '') or _ANTHROPIC_API_KEY
try:
    resp = _req.post(
        'https://api.anthropic.com/v1/messages',
        headers={
            'x-api-key': api_key,
            'anthropic-version': '2023-06-01',
            'content-type': 'application/json',
        },
        json={
            'model': 'claude-sonnet-4-20250514',
            'max_tokens': 1024,
            'messages': [{'role': 'user', 'content': 'Say: API connection confirmed.'}],
        },
        timeout=60,
    )
    print(f"  Direct API test status: {resp.status_code}")
    if resp.status_code != 200:
        print(f"  Error body: {resp.text[:300]}")
except Exception as e:
    print(f"  Direct API test exception: {e}")

ai_summary = ai_summarize(engine.state, template)

check("ai_summarize returns non-empty string",
      len(ai_summary) > 50,
      f"got: {ai_summary[:100]}")
check("AI summary is not just the fallback HPI",
      ai_summary != template.get('hpi', ''),
      f"got fallback: {ai_summary[:100]}")

print("\n--- AI Summary (Sonnet) ---")
print(ai_summary)


Calling Anthropic API for AI summary...
  API key loaded from api_keys.py: True (starts with: sk-ant-api03...)
  Direct API test status: 200
  ✅ ai_summarize returns non-empty string
  ✅ AI summary is not just the fallback HPI

--- AI Summary (Sonnet) ---
This 90-minute history of constant, crushing central chest pain with heavy pressure quality began at rest and is rated 9/10 in severity. The pain radiates to the left arm and jaw and has been progressively worsening over the 90-minute duration. Associated symptoms include shortness of breath and diaphoresis, with the patient unable to perform any activities and lying still due to functional impairment. The pain shows no relief with rest or nitroglycerin, with nothing providing any amelioration. The patient has a significant cardiac risk profile including hypertension and hypercholesterolemia, along with a family history of early coronary artery disease (father with MI at age 55). The patient denies syncope, near-syncope, palpitations,

## Summary

In [21]:
print("\n" + "="*50)
print(f"RESULTS: {len(passed)} passed, {len(failed)} failed")
print("="*50)
if failed:
    print("\nFailed tests:")
    for f in failed:
        print(f"  ❌ {f}")
else:
    print("\nAll tests passed ✅")


RESULTS: 77 passed, 0 failed

All tests passed ✅
